In [ ]:
%load_ext autoreload
%autoreload 2
%pdb on # clickable err traceback



from __future__ import absolute_import, division, print_function
import torch
from trainer_endoda3 import Trainer
from options_endoda3 import MonodepthOptions


In [ ]:
# Minimal options for testing
options = MonodepthOptions()
args = [
    '--batch_size', '8',
    '--batch_size', '2',
    '--batch_size', '1',
    '--num_workers', '0',
    '--of_samples',
    '--of_samples_num', '1',
    '--of_samples_num', '10',
    '--frame_ids', '0', '-1', '1',
    '--train_frame_ids', '0', '-5', '5',
    '--train_frame_ids', '0', '-1', '1',
    '--dataset', 'endovis',
    '--data_path', '/mnt/cluster/workspaces/jinjingxu/SCARED_Images_Resized/',
    '--log_dir', '/tmp/endoda_debug',
    '--log_dir', '/mnt/cluster/workspaces/jinjingxu/tmp/',
    '--compute_depth_metrics',
    '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-all-wowrapper.yaml',
    '--pose_model_type', 'da3_internal',
    '--k_model_type', 'da3_internal',
    '--of_supervised_with_which', 'inputs_color',
    '--learn_intrinsics',
    '--train_data_file', 'test_files.txt',
    '--train_data_file', 'test_files_sequence1_val.txt',
    '--train_data_file', 'train_files.txt',
    '--val_data_file', 'test_files.txt test_files_sequence1_val.txt  test_files_sequence2_val.txt ',
    '--val_data_file', 'test_files.txt',
    '--val_data_file', 'test_files_sequence1_val.txt',
    '--da3_depth_regression_target', 'disp',
    '--da3_depth_regression_target', 'disp',
    '--da3_depth_regression_target', 'depth2disp',
    '--da3_depth_regression_target', 'depth2disp_v2',
    '--depth_model_type', 'endodac',
    '--depth_model_type', 'depthanything3',
    '--k_model_type', 'mlp_with_pn_bottleneck_ipt',
    '--k_model_type', 'da3_internal',
    '--warm_up_step', '5000',
    '--pose_model_type', 'separate_resnet',
    '--pretrained_path', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/weights/depthanything',
    '--pretrained_path', 'depth-anything/da3-base',
    '--af_model_type', 'adjust_net',
    '--of_model_type', 'raft',
    # '--depth_model_type', 'endodac',

# --depth_model_type endodac --da3_depth_regression_target disp --k_model_type mlp_with_pn_bottleneck_ipt --warm_up_step 5000 --pretrained_path /mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/weights/depthanything \


    # '--enable_seq_inputs',
    # '--of_model_type', 'raft',
    # '--use_raft_multi_iters',
    # '--raft_trainable_modules', 'convnormrelu layer1 layer2_0 ',
    # '--learn_intrinsics',
    # '--use_perframe_gt_K',
 
    # '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-depth-wowrapper.yaml'
]
opts = options.parse_notebook(args)


# Initialize trainer
trainer = Trainer(opts)
print(f"Trainer initialized on {trainer.device}")


In [ ]:
# # loop over trainer.val_loader to check the collate_fn
# for i, val_inputs in enumerate(trainer.val_loader):
#     print("Batch", i)

In [ ]:
# Get sample batch
trainer.step = 0
trainer.set_train()
trainer.current_frame_ids = trainer.train_frame_ids  # Set context for training
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)
val_iter = iter(trainer.val_loader)
val_inputs = next(val_iter)

# Forward pass
outputs, losses = trainer.process_batch(inputs)

print("Forward pass completed!")
print(f"Output keys: {len(outputs)} keys")
print(f"Loss: {losses['loss'].item():.6f}")
print(f"Loss components: {list(losses.keys())}")


In [ ]:
# Compute losses explicitly
# Ensure current_frame_ids is set for training context
trainer.current_frame_ids = trainer.train_frame_ids
losses = trainer.compute_losses(inputs, outputs)

print("Loss computation:")
for key, val in losses.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")

# compute losses_0 explicitly
losses_0 = trainer.compute_losses_0(inputs, outputs)

print("Loss computation_0:")
for key, val in losses_0.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")

In [ ]:
# optional: obtain various image from outputs and save as one row of images of all image
from utils import img_gen
for key in outputs.keys():
    outputs[key] = outputs[key].detach()
    # print(f"  {key}: shape={outputs[key].shape}")
# compute the depth_err metrics    
from utils.metrics import compute_depth_metrics
metrics = compute_depth_metrics(inputs, outputs)
merged_dict = {**inputs, **outputs, **metrics}
img_gen(
    merged_dict=merged_dict,
    image_keys_row1=[
    ("color", 0, 0), 
    ("color_aug", 0, 0), 
    ("color", -1, 0), 
    ("color_aug", -1, 0), 
    # ("disp", 0), 
    ("depth", 0, 0), 
    ("depth_gt", 0, 0), 
    ("occu_mask_backward", 0, -1), 
    ("occu_mask_backward", 1, -1), 
    ("occu_mask_backward", 3, -1), 

    ("pose_flow_dbg", "high", -1, 3), 
    ("position", "high", 3, -1), 
    
    # "depth_err",
    # "depth_err",
    ],
    # image_keys_row2=[
    #     ("color_aug", 0, 0),  # Target
    #     ("color_warp", 0, -1),  # Raw warped
    #     ("paba_color_warp", 0, -1),  # Aligned
    #     ("paba_alpha", 0, -1),  # Alpha map
    #     "color_warp_err_before",
    #     "color_warp_err_after_afstyle_color_warp",
    # ],
    save_path="output_grid.png",
    sample_idx=0
)

In [ ]:
# Set context for validation
trainer.current_frame_ids = trainer.val_frame_ids
val_outputs, _ = trainer.process_batch_val(val_inputs)
val_losses = trainer.compute_losses_val(val_inputs, val_outputs)

In [ ]:
# Get validation batch (has GT depth and poses)
trainer.set_eval()
trainer.current_frame_ids = trainer.val_frame_ids  # Set context for validation
val_iter = iter(trainer.val_loader)
val_inputs = next(val_iter)

# for flag in [True, False]:
    # trainer.replace_with_gt_rel_rotation = flag
    # print('replace_with_gt_rel_rotation: ', flag)

# Forward pass
with torch.no_grad():
    val_outputs, val_losses = trainer.process_batch_val(val_inputs)

# Compute depth metrics
from utils.metrics import compute_depth_metrics, compute_pose_metrics

depth_metrics = compute_depth_metrics(val_inputs, val_outputs)
# Use current_frame_ids which is already set to val_frame_ids
pose_metrics = compute_pose_metrics(val_inputs, val_outputs, trainer.current_frame_ids)

print("Depth Metrics:")
if depth_metrics:
    for key, val in depth_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No depth metrics (GT depth not available)")

print("\nPose Metrics:")
print(pose_metrics)
if pose_metrics:
    for key, val in pose_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No pose metrics (GT poses not available)")


In [ ]:
# Full training step
trainer.set_train()
trainer.current_frame_ids = trainer.train_frame_ids  # Set context for training
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)

# Forward
outputs, losses = trainer.process_batch(inputs)

# Backward
trainer.model_optimizer.zero_grad()
losses["loss"].backward()
trainer.model_optimizer.step()

print(f"Training step completed! Loss: {losses['loss'].item():.6f}")
